# Data Storytelling: disaster patterns and impact

This notebook organizes the analysis into a visual and progressive narrative:
- Where disasters occur most frequently,
- Which disaster types predominate,
- And how these events relate to deaths and economic damage.

The central idea is to turn the data into a clear story about risk, severity, and prioritization of action.


## Dataset overview

This cell shows the scale of the problem and the first aggregates, such as the number of records, columns, and regional distribution. This step helps set the context before diving deeper into the patterns.

In [100]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv("../data/prepared_public_emdat_2026.csv")

print(f"Number of rows: {len(df)}")
print(f"Available columns: {len(df.columns)}")
print("\nAvailable regions:")
print(df["Region"].value_counts().to_string())

region_summary = (
    df.groupby("Region")
      .agg(
          events=("DisNo.", "count"),
          deaths=("Total Deaths", "sum"),
          damage=("Total Damage ('000 US$)", "sum")
      )
      .reset_index()
)

region_summary

Number of rows: 16769
Available columns: 33

Available regions:
Region
Asia        6761
Africa      4306
Americas    3339
Europe      1951
Oceania      412


,Region,events,deaths,damage
0,Africa,4306,248004,5.459985e+07
1,Americas,3339,304190,1.982726e+09
2,Asia,6761,979180,1.448219e+09
3,Europe,1951,344398,3.411777e+08
4,Oceania,412,4625,1.013602e+08


## Frequency of events by region

This cell addresses the question: where do disasters occur most frequently? The chart shows the geographic concentration of events and already suggests where the risk is higher.

In [101]:
fig = px.bar(
    region_summary.sort_values("events", ascending=False),
    x="Region",
    y="events",
    text="events",
    title="Number of events by region"
)

fig.update_traces(texttemplate="%{text}", textposition="outside")
fig.update_layout(xaxis_title="Region", yaxis_title="Number of events")
fig.show()

## Most frequent disaster types

In this stage, the analysis shifts focus to the nature of the events. The goal is to identify which disaster types appear most recurrently in the dataset.

In [102]:
type_summary = (
    df.groupby("Disaster Type")
      .agg(
          events=("DisNo.", "count"),
          deaths=("Total Deaths", "sum"),
          damage=("Total Damage ('000 US$)", "sum")
      )
      .reset_index()
      .sort_values("events", ascending=False)
      .head(10)
)

type_summary

,Disaster Type,events,deaths,damage
13,Flood,4256,141913,8.296867e+08
27,Storm,2849,227367,1.928891e+09
26,Road,2245,48438,0.000000e+00
29,Water,1181,53030,0.000000e+00
7,Epidemic,893,125262,0.000000e+00
6,Earthquake,695,796373,6.677752e+08
10,Extreme temperature,568,352756,5.286719e+07
20,Mass movement (wet),499,23116,5.458231e+06
8,Explosion (Industrial),474,14433,3.663940e+07
12,Fire (Miscellaneous),455,11400,1.785576e+06


## Distribution of disaster types

This chart complements the previous table by visually showing which disaster types dominate the dataset.

In [103]:
fig = px.bar(
    type_summary,
    x="Disaster Type",
    y="events",
    text="events",
    color="Disaster Type",
    title="Most frequent disaster types"
)

fig.update_traces(texttemplate="%{text}", textposition="outside")
fig.update_layout(xaxis_title="Disaster type", yaxis_title="Number of events", showlegend=False)
fig.show()

## Affected people by disaster subgroup over time

This section examines how the number of people affected evolves over time across disaster subgroups. It helps connect the frequency of events with the human impact they generate.


In [104]:
affected_over_time = (
    df.groupby(["Start Year", "Disaster Subgroup"])
      .agg(
          events=("DisNo.", "count"),
          affected_people=("Total Affected", "sum")
      )
      .reset_index()
)

fig = px.line(
    affected_over_time,
    x="Start Year",
    y="affected_people",
    color="Disaster Subgroup",
    markers=True,
    title="Affected people by disaster subgroup over time"
)

fig.update_xaxes(title="Year")
fig.update_yaxes(title="Affected people")
fig.show()


## Relationship between deaths, region, and disaster type

This cell investigates whether certain disaster types in certain regions tend to cause more deaths and damage. To do this, we aggregate the data by region and disaster type and apply a logarithmic scale to reduce the effect of extreme values.

The chart suggests a positive relationship between deaths and damage: when a disaster type in a region causes more deaths, there also tend to be greater economic losses. This indicates that the severity of the event is not only a human problem, but also a problem of material and institutional impact.

In [105]:
impact_summary = (
    df.groupby(["Region", "Disaster Type"])
      .agg(
          events=("DisNo.", "count"),
          deaths=("Total Deaths", "sum"),
          damage=("Total Damage ('000 US$)", "sum")
      )
      .reset_index()
)

impact_summary["deaths_log"] = np.log1p(impact_summary["deaths"])
impact_summary["damage_log"] = np.log1p(impact_summary["damage"])

fig = px.scatter(
    impact_summary,
    x="deaths_log",
    y="damage_log",
    color="Region",
    size="events",
    hover_name="Disaster Type",
    title="Relation between deaths and damages by region and disaster type"
)

fig.update_xaxes(title="Deaths (log1p)")
fig.update_yaxes(title="Damage (log1p)")
fig.show()

impact_summary.sort_values(["deaths", "damage"], ascending=False).head(10)

,Region,Disaster Type,events,deaths,damage,deaths_log,damage_log
58,Asia,Earthquake,470,559523,5.236536e+08,13.234842,20.076341
91,Europe,Extreme temperature,322,324839,2.001105e+07,12.691088,16.811795
31,Americas,Earthquake,112,229087,6.076747e+07,12.341861,17.922565
78,Asia,Storm,1153,187905,3.465151e+08,12.143697,19.663437
7,Africa,Epidemic,615,100402,0.000000e+00,11.516947,0.000000
65,Asia,Flood,1727,94808,4.770807e+08,11.459620,19.983196
13,Africa,Flood,1019,28304,1.678646e+07,10.250794,16.636083
24,Africa,Water,526,25186,0.000000e+00,10.134083,0.000000
5,Africa,Drought,178,23374,6.438261e+06,10.059422,15.677769
62,Asia,Extreme temperature,155,21918,2.650133e+07,9.995109,17.092706


## Final insight and conclusion

This final cell brings together the findings in a summary. It helps close the narrative with a clear message: severity is not uniform, and some contexts deserve greater attention.

In [106]:
top_regions = region_summary.sort_values("deaths", ascending=False).head(5)
top_disasters = type_summary.head(5)

print("Top regions by deaths:")
print(top_regions[["Region", "deaths", "damage", "events"]].to_string(index=False))

print("\nTop disaster types by frequency:")
print(top_disasters[["Disaster Type", "events", "deaths", "damage"]].to_string(index=False))

Top regions by deaths:
  Region  deaths       damage  events
    Asia  979180 1.448219e+09    6761
  Europe  344398 3.411777e+08    1951
Americas  304190 1.982726e+09    3339
  Africa  248004 5.459985e+07    4306
 Oceania    4625 1.013602e+08     412

Top disaster types by frequency:
Disaster Type  events  deaths       damage
        Flood    4256  141913 8.296867e+08
        Storm    2849  227367 1.928891e+09
         Road    2245   48438 0.000000e+00
        Water    1181   53030 0.000000e+00
     Epidemic     893  125262 0.000000e+00
